In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    (
        path
        for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
        if (path / "config.py").is_file()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find config.py. Start Jupyter from the repository or its Notebooks folder."
    )
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from config import (
    DAILY_FLOW_FILE,
    PRODUCT_RISK_FILE,
    KPI_WEEK_FREQUENCY,
    OPERATIONAL_RISK_DAYS,
    WEEKLY_FLOW_FILE,
    ensure_output_directories,
)

df = pd.read_csv(DAILY_FLOW_FILE)
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df = df.sort_values(["product_id", "Date"]).reset_index(drop=True)
df.columns


In [ ]:
numeric_cols = [
    "sales_units",
    "production_units",
    "delivery_units",
    "factory_units",
]

weekly = (
    df.assign(
        week_start=df["Date"]
        .dt.to_period(KPI_WEEK_FREQUENCY)
        .dt.to_timestamp()
    )
    .groupby(
        ["product_id", "week_start", "Group", "Sub-Group"],
        dropna=False,
    )[numeric_cols]
    .sum()
    .reset_index()
)


In [ ]:
weekly["demand_production_gap"]= weekly["sales_units"] - weekly["production_units"]
weekly["demand_delivery_gap"]= weekly["sales_units"] - weekly["delivery_units"]
weekly["factory_delivery_gap"]= weekly["factory_units"] - weekly["delivery_units"]

In [ ]:
weekly["production_to_demand_ratio"] = np.where(
    weekly["sales_units"] > 0,
    weekly["production_units"] / weekly["sales_units"],
    np.nan,
)
weekly["delivery_to_demand_ratio"] = np.where(
    weekly["sales_units"] > 0,
    weekly["delivery_units"] / weekly["sales_units"],
    np.nan,
)


In [ ]:
overall_weekly = weekly.groupby("week_start", as_index=False)[numeric_cols].sum()

overall_weekly.plot(
    x="week_start",
    y=["sales_units", "production_units", "delivery_units"],
    figsize=(12, 6),
)
plt.title("Weekly Demand, Production and Delivery")
plt.ylabel("Units")
plt.xlabel("Week")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


buidling a risk monitor

In [ ]:
end_date = df["Date"].max()
start_date = end_date - pd.Timedelta(days=OPERATIONAL_RISK_DAYS - 1)
recent = df[df["Date"].between(start_date, end_date)]

product_risk = recent.groupby("product_id", as_index=False)[numeric_cols].sum()
product_risk["production_gap"] = (
    product_risk["sales_units"] - product_risk["production_units"]
)
product_risk["delivery_gap"] = (
    product_risk["sales_units"] - product_risk["delivery_units"]
)
product_risk["production_ratio"] = np.where(
    product_risk["sales_units"] > 0,
    product_risk["production_units"] / product_risk["sales_units"],
    np.nan,
)
product_risk["delivery_ratio"] = np.where(
    product_risk["sales_units"] > 0,
    product_risk["delivery_units"] / product_risk["sales_units"],
    np.nan,
)


In [ ]:
conditions = [
    (
        (product_risk["production_gap"] > 0)
        & (product_risk["delivery_gap"] > 0)
    ),
    (
        (product_risk["production_gap"] > 0)
        | (product_risk["delivery_gap"] > 0)
    )
]

choices = ["High", "Medium"]

product_risk["risk_level"] = np.select(
    conditions,
    choices,
    default="Low"
)

In [ ]:
product_risk.sort_values(
    "delivery_gap",
    ascending=False
).head(10)

In [ ]:
demand_profile = (
    df.groupby("product_id")
    .agg(
        total_demand=("sales_units", "sum"),
        average_daily_demand=("sales_units", "mean"),
        demand_std=("sales_units", "std"),
        zero_demand_days=("sales_units", lambda values: (values == 0).sum()),
        observed_days=("Date", "nunique"),
    )
    .reset_index()
)
demand_profile["zero_demand_rate"] = (
    demand_profile["zero_demand_days"] / demand_profile["observed_days"]
)
demand_profile["demand_cv"] = (
    demand_profile["demand_std"] / demand_profile["average_daily_demand"]
)


In [ ]:
weekly_columns = [
    "product_id",
    "week_start",
    "Group",
    "Sub-Group",
    "sales_units",
    "production_units",
    "delivery_units",
    "factory_units",
    "production_to_demand_ratio",
    "delivery_to_demand_ratio",
    "demand_production_gap",
    "demand_delivery_gap",
    "factory_delivery_gap",
]
weekly = weekly[weekly_columns]

ensure_output_directories()
weekly.to_csv(WEEKLY_FLOW_FILE, index=False)
product_risk.to_csv(PRODUCT_RISK_FILE, index=False)
print(f"Saved: {WEEKLY_FLOW_FILE}")
print(f"Saved: {PRODUCT_RISK_FILE}")
